<a href="https://colab.research.google.com/github/Nakib-Nasrullah/Heart_disease/blob/main/final_defense.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pandas==2.2.2 wfdb==3.4.1 scipy matplotlib

import wfdb
import numpy as np
import pandas as pd
import os

In [2]:
DATA_DIR = "mitdb"

if not os.path.exists(DATA_DIR):
    wfdb.dl_database('mitdb', dl_dir=DATA_DIR)

print("Dataset ready!")

Generating record list for: 100
Generating record list for: 101
Generating record list for: 102
Generating record list for: 103
Generating record list for: 104
Generating record list for: 105
Generating record list for: 106
Generating record list for: 107
Generating record list for: 108
Generating record list for: 109
Generating record list for: 111
Generating record list for: 112
Generating record list for: 113
Generating record list for: 114
Generating record list for: 115
Generating record list for: 116
Generating record list for: 117
Generating record list for: 118
Generating record list for: 119
Generating record list for: 121
Generating record list for: 122
Generating record list for: 123
Generating record list for: 124
Generating record list for: 200
Generating record list for: 201
Generating record list for: 202
Generating record list for: 203
Generating record list for: 205
Generating record list for: 207
Generating record list for: 208
Generating record list for: 209
Generati

In [3]:
WINDOW = 187
HALF = WINDOW // 2

label_map = {
    'N': 0, 'L': 0, 'R': 0, 'e': 0, 'j': 0,
    'A': 1, 'a': 1, 'J': 1, 'S': 1,
    'V': 2, 'E': 2,
    'F': 3
}

beats = []

records = sorted([f.split('.')[0] for f in os.listdir(DATA_DIR) if f.endswith('.dat')])

for record in records:
    try:
        signal, _ = wfdb.rdsamp(os.path.join(DATA_DIR, record))
        ann = wfdb.rdann(os.path.join(DATA_DIR, record), 'atr')
    except:
        continue

    ecg = signal[:, 0]

    for r, sym in zip(ann.sample, ann.symbol):
        if sym not in label_map:
            continue

        if r - HALF < 0 or r + HALF >= len(ecg):
            continue

        beat = ecg[r-HALF:r+HALF+1]
        beats.append([record] + beat.tolist() + [label_map[sym]])

columns = ["record_id"] + [f"f{i}" for i in range(WINDOW)] + ["label"]
df = pd.DataFrame(beats, columns=columns)

df.to_csv("mitbih_patient_level.csv", index=False)

print("Dataset created:", df.shape)

Dataset created: (101426, 189)


In [4]:
from sklearn.model_selection import train_test_split

df = pd.read_csv("mitbih_patient_level.csv")

patients = df['record_id'].unique()

# 80% train+val, 20% test
trainval_patients, test_patients = train_test_split(
    patients,
    test_size=0.20,
    random_state=42
)

# 10% of 80% → validation
train_patients, val_patients = train_test_split(
    trainval_patients,
    test_size=0.10,
    random_state=42
)

train_df = df[df['record_id'].isin(train_patients)]
val_df   = df[df['record_id'].isin(val_patients)]
test_df  = df[df['record_id'].isin(test_patients)]

# Check
assert set(train_df['record_id']).isdisjoint(val_df['record_id'])
assert set(train_df['record_id']).isdisjoint(test_df['record_id'])
assert set(val_df['record_id']).isdisjoint(test_df['record_id'])

print("No patient overlap ✔")

# Save
train_df.to_csv("mitbih_train.csv", index=False)
val_df.to_csv("mitbih_val.csv", index=False)
test_df.to_csv("mitbih_test.csv", index=False)

print("Split done!")

No patient overlap ✔
Split done!


In [5]:
train_df = pd.read_csv("mitbih_train.csv")
val_df   = pd.read_csv("mitbih_val.csv")
test_df  = pd.read_csv("mitbih_test.csv")

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

Train: (72737, 189)
Val: (8245, 189)
Test: (20444, 189)


In [6]:
X_train = train_df.iloc[:, 1:-1].values
y_train = train_df['label'].values.astype(int)

X_val = val_df.iloc[:, 1:-1].values
y_val = val_df['label'].values.astype(int)

X_test = test_df.iloc[:, 1:-1].values
y_test = test_df['label'].values.astype(int)